# Boop — KataGo-Style AlphaZero Training

Trains a **CNN ResNet policy+value network** with MCTS self-play,
incorporating key improvements from KataGo:

| Improvement | Vanilla AlphaZero | This notebook |
|---|---|---|
| Network | MLP (3×256) | **6-block ResNet + SE attention**, CNN on 6×6 board |
| Input | Flat 184-float obs | **9-channel (5 board + 4 hand)** 6×6 spatial tensor |
| Training data | 1× per move | **8× via all 8 board symmetries** |
| MCTS sims | Fixed | **Playout cap randomization** (15 fast / 100 full) |
| Temperature | Constant | **Annealing** — 1.0 first 30 moves, 0.05 after |
| LR | Cosine from ep 1 | **Linear warmup** (50 eps) then cosine decay |
| Optimizer | Adam | **AdamW** (true weight decay) |
| Entropy | None | **Entropy bonus** (prevents early mode collapse) |

### Network architecture
```
Input (9, 6, 6)  →  Stem (Conv 3×3, BN, ReLU)
                 →  6 × ResBlock (Conv-BN-ReLU-Conv-BN + SE) + skip
                 →  Policy head: 1×1 conv (2 ch) → flatten → Linear(72)
                 →  Value head:  1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
```

### KataGo improvements in detail
- **SE attention**: per-channel gating learns which feature maps matter for each position
- **Symmetry augmentation**: 4 rotations × 2 reflections = 8× free training data
- **Playout cap randomization**: 75% fast (15 sim) games for diversity + 25% full (100 sim) games for quality
- **Temperature annealing per game**: exploratory early, near-greedy for endgame decisions
- **LR warmup**: avoids destabilising BatchNorm layers with large early gradients
- **Entropy bonus**: small regulariser that keeps the policy spread out early in training

In [ ]:
%pip install open_spiel -q

In [ ]:
# Boop board game -- inlined so Colab works without the custom package.
# Matches open_spiel/python/games/boop.py (includes stuck-state fix).

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_ACTIONS = _NUM_PIECE_TYPES * _NUM_CELLS  # 72
_MAX_KITTENS = 8
_MAX_CATS = 6
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    actions = []
    for r in range(_ROWS):
      for c in range(_COLS):
        if self.board[r, c] == _EMPTY:
          cell = r * _COLS + c
          if self._hand[player][0] > 0:
            actions.append(cell)
          if self._hand[player][1] > 0:
            actions.append(_NUM_CELLS + cell)
    return sorted(actions)

  def _apply_action(self, action):
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    p = self._cur_player
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # Real Boop rule: skip the next player's turn if they have no pieces.
    # Only declare a draw if BOTH players are simultaneously out of pieces.
    if not self._legal_actions(self._cur_player):
      self._cur_player = p  # stay with current player
      if not self._legal_actions(self._cur_player):
        self._is_terminal = True  # both stuck → draw

  def _action_to_string(self, player, action):
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = self.board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor in (_P0_CAT, _P1_CAT)
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if neighbor in (_P0_KITTEN, _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          self.board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif self.board[dest_r, dest_c] == _EMPTY:
          self.board[dest_r, dest_c] = neighbor
          self.board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    kitten_val = _KITTEN_VAL[player]
    cat_val = _CAT_VAL[player]
    to_promote = set()
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          cells = []
          for k in range(3):
            nr, nc = r + dr * k, c + dc * k
            if (0 <= nr < _ROWS and 0 <= nc < _COLS
                and self.board[nr, nc] == kitten_val):
              cells.append((nr, nc))
            else:
              break
          if len(cells) == 3:
            to_promote.update(cells)
    if not to_promote:
      return
    n = len(to_promote)
    cats_on_board = int(np.sum(self.board == cat_val))
    for pr, pc in to_promote:
      self.board[pr, pc] = _EMPTY
      self._hand[player][0] += 1
    cats_to_add = min(n, max(0, _MAX_CATS - cats_on_board - self._hand[player][1]))
    self._hand[player][1] += cats_to_add

  def _check_win(self, player):
    cat_val = _CAT_VAL[player]
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          if all(
              0 <= r + dr * k < _ROWS
              and 0 <= c + dc * k < _COLS
              and self.board[r + dr * k, c + dc * k] == cat_val
              for k in range(3)):
            return True
    return False


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    opp = 1 - player
    mk, mc = _KITTEN_VAL[player], _CAT_VAL[player]
    ok, oc = _KITTEN_VAL[opp], _CAT_VAL[opp]
    for r in range(_ROWS):
      for c in range(_COLS):
        v = state.board[r, c]
        if v == _EMPTY:   obs[0, r, c] = 1.0
        elif v == mk:     obs[1, r, c] = 1.0
        elif v == mc:     obs[2, r, c] = 1.0
        elif v == ok:     obs[3, r, c] = 1.0
        elif v == oc:     obs[4, r, c] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)

In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Actions:', game.num_distinct_actions())
print('Obs size:', game.observation_tensor_size())
print()
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


# ── Input helpers ──────────────────────────────────────────────────────────────────────────────

def _obs_to_9ch(obs_np):
    """Flat 184-float obs → (9, 6, 6): 5 board planes + 4 hand scalars broadcast."""
    board = obs_np[..., :180].reshape(*obs_np.shape[:-1], 5, 6, 6)
    hand  = obs_np[..., 180:]
    # broadcast hand scalars spatially so the CNN sees them at every cell
    hand_planes = np.broadcast_to(
        hand[..., None, None], hand.shape + (6, 6)).copy()
    return np.concatenate([board, hand_planes], axis=-3)   # (..., 9, 6, 6)


def state_to_tensor(state, device):
    """Single game state → (1, 9, 6, 6) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = _obs_to_9ch(obs)[None]        # (1, 9, 6, 6)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat 184-float observations → (B, 9, 6, 6) float tensor."""
    obs = np.stack(obs_list).astype(np.float32)   # (B, 184)
    x   = _obs_to_9ch(obs)                         # (B, 9, 6, 6)
    return torch.from_numpy(x).to(device)


# ── Network modules ────────────────────────────────────────────────────────────────────────────

class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))             # global avg pool → (B, C)
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-BN-ReLU-Conv-BN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class BoopNet(nn.Module):
    """
    KataGo-style network for Boop.

    Input  : (B, 9, 6, 6) — 5 board planes + 4 hand scalars broadcast
    Body   : Conv stem → N × ResBlock(channels, SE)
    Policy : 1×1 conv (2 ch) → flatten → Linear(72)
    Value  : 1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
    """
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(9, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 6 * 6, 72),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(36, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.body(self.stem(x))
        return self.policy_head(x), self.value_head(x)


print(f'Device: {DEVICE}')
_demo = BoopNet()
print(f'BoopNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
from open_spiel.python.algorithms import mcts as mcts_lib


# ── 8-fold board symmetry augmentation ──────────────────────────────────────────────────────────────────────
# The 6×6 Boop board has 8 dihedral symmetries (4 rotations × 2 reflections).
# Every self-play sample is augmented with all 8 copies for free training data.

def _compute_action_perm(sym_idx):
    """
    Precompute the 72-element action permutation for board symmetry sym_idx.
    Symmetries 0-3: 0°/90°/180°/270° CCW rotation.
    Symmetries 4-7: horizontal flip followed by the same rotations.
    """
    k    = sym_idx % 4
    flip = sym_idx >= 4
    perm = np.empty(72, dtype=np.int32)
    for pt in range(2):
        for r in range(6):
            for c in range(6):
                nr, nc = r, c
                if flip:
                    nc = 5 - nc
                for _ in range(k):       # k × 90° CCW: (r,c) → (5-c, r)
                    nr, nc = 5 - nc, nr
                perm[pt * 36 + r * 6 + c] = pt * 36 + nr * 6 + nc
    return perm

_ACTION_PERMS = [_compute_action_perm(i) for i in range(8)]


def augment_sample(obs_flat, policy, value):
    """Return all 8 symmetry copies of a (obs, policy, value) training sample."""
    obs   = np.asarray(obs_flat, dtype=np.float32)
    board = obs[:180].reshape(5, 6, 6)
    hand  = obs[180:]                       # hand counts: spatially invariant
    pol   = np.asarray(policy, dtype=np.float32)

    out = []
    for sym_idx in range(8):
        k    = sym_idx % 4
        flip = sym_idx >= 4
        b    = board.copy()
        if flip:
            b = b[:, :, ::-1]             # horizontal flip (cols)
        b = np.rot90(b, k=k, axes=(1, 2)) # k × 90° CCW
        aug_obs = np.concatenate([b.reshape(-1).copy(), hand])
        out.append((aug_obs, pol[_ACTION_PERMS[sym_idx]], value))
    return out


# ── MCTS evaluator ───────────────────────────────────────────────────────────────────────────────

class NNEvaluator(mcts_lib.Evaluator):
    """
    Wraps BoopNet as an OpenSpiel MCTS evaluator.
    evaluate() → list of returns per player  (MCTS indexes by player)
    prior()    → (action, prob) pairs from policy head
    """
    def __init__(self, network, device=DEVICE):
        self.network = network
        self.device  = device

    def evaluate(self, state):
        with torch.no_grad():
            _, value = self.network(state_to_tensor(state, self.device))
        v   = value.item()
        cur = state.current_player()
        ret = [-v, -v]
        ret[cur] = v
        return ret

    def prior(self, state):
        legal = state.legal_actions()
        if not legal:
            return []
        with torch.no_grad():
            logits, _ = self.network(state_to_tensor(state, self.device))
        logits = logits.squeeze().cpu().numpy()
        lg     = logits[legal] - logits[legal].max()
        probs  = np.exp(lg)
        probs /= probs.sum()
        return list(zip(legal, probs.tolist()))


def make_bot(game, evaluator, num_simulations):
    return mcts_lib.MCTSBot(
        game,
        uct_c=1.4,
        max_simulations=num_simulations,
        evaluator=evaluator,
        dirichlet_noise=(0.25, 0.3),
    )


# ── Self-play ───────────────────────────────────────────────────────────────────────────────────

def self_play_game(game, bot, temp_threshold=30):
    """
    MCTS self-play with temperature annealing:
      moves < temp_threshold : temperature = 1.0  (explore)
      moves >= temp_threshold: temperature = 0.05 (near-greedy for endgame)
    Returns list of (obs_flat, policy_vec, game_return) per move.
    """
    state    = game.new_initial_state()
    history  = []
    move_num = 0

    while not state.is_terminal():
        player      = state.current_player()
        obs         = state.observation_tensor(player)
        temperature = 1.0 if move_num < temp_threshold else 0.05

        root      = bot.mcts_search(state)
        legal     = state.legal_actions()
        visit_map = {ch.action: ch.explore_count for ch in root.children}
        counts    = np.array([visit_map.get(a, 0) for a in legal], dtype=np.float32)

        if temperature < 1e-2 or counts.sum() == 0:
            probs          = np.zeros(len(legal), dtype=np.float32)
            probs[counts.argmax()] = 1.0
        else:
            ct    = counts ** (1.0 / temperature)
            probs = ct / ct.sum()

        policy_vec = np.zeros(72, dtype=np.float32)
        for a, p in zip(legal, probs):
            policy_vec[a] = p

        action = np.random.choice(legal, p=probs)
        history.append((player, list(obs), policy_vec))
        state.apply_action(action)
        move_num += 1

    returns = state.returns()
    return [(obs, pol, returns[pl]) for pl, obs, pol in history]


# ── Evaluation ──────────────────────────────────────────────────────────────────────────────────

def greedy_action(network, state, device=DEVICE, temperature=0.0):
    with torch.no_grad():
        logits, _ = network(state_to_tensor(state, device))
    logits = logits.squeeze().cpu().numpy()
    legal  = state.legal_actions()
    if temperature == 0.0:
        return legal[int(logits[legal].argmax())]
    lg = logits[legal] - logits[legal].max()
    probs = np.exp(lg / temperature)
    probs /= probs.sum()
    return int(np.random.choice(legal, p=probs))


def eval_vs_random(game, network, num_games=100, device=DEVICE):
    wins = 0
    for _ in range(num_games):
        state = game.new_initial_state()
        while not state.is_terminal():
            if state.current_player() == 0:
                action = greedy_action(network, state, device)
            else:
                action = random.choice(state.legal_actions())
            state.apply_action(action)
        if state.returns()[0] > 0:
            wins += 1
    return wins / num_games


def eval_vs_snapshot(game, network, snapshot, num_games=100, device=DEVICE):
    wins = 0
    for i in range(num_games):
        state = game.new_initial_state()
        net_player = i % 2  # alternate: net plays P0 in even games, P1 in odd
        while not state.is_terminal():
            net    = network if state.current_player() == net_player else snapshot
            action = greedy_action(net, state, device, temperature=1.0)
            state.apply_action(action)
        r = state.returns()
        if r[net_player] > 0:
            wins += 1
    return wins / num_games


def update_elo(elo, opp_elo, win_rate, k=32):
    expected = 1.0 / (1.0 + 10 ** ((opp_elo - elo) / 400.0))
    return elo + k * (win_rate - expected)

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────────────────────
# Colab GPU estimate: ~2-4 h for 2000 episodes with these settings.
# Reduce FULL_MCTS_SIMS or NUM_EPISODES to go faster.
NUM_EPISODES     = 2_000
FULL_MCTS_SIMS   = 100     # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 15      # fast self-play for diverse data (75% of games)
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
TEMP_THRESHOLD   = 30      # move number to switch from temp=1.0 to temp=0.05
EVAL_EVERY       = 100
EVAL_GAMES       = 50
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 4
MAX_BUFFER       = 200_000  # 8× augmentation inflates buffer; cap it
CHANNELS         = 128
NUM_BLOCKS       = 6
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 50       # linear warmup episodes
WEIGHT_DECAY     = 1e-4
ENTROPY_BONUS    = 0.01     # policy entropy regularisation coefficient
RANDOM_ELO       = 600.0

# ── Setup ──────────────────────────────────────────────────────────────────────────────────────
game      = pyspiel.load_game('python_boop')
network   = BoopNet(CHANNELS, NUM_BLOCKS).to(DEVICE)
optimizer = torch.optim.AdamW(network.parameters(),
                               lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = (ep - LR_WARMUP_EPS) / max(NUM_EPISODES - LR_WARMUP_EPS, 1)
    return 0.5 * (1.0 + np.cos(np.pi * frac))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

# Both bots share one evaluator (and thus the same network reference).
# Weights updated in-place → both bots always use the latest network.
evaluator = NNEvaluator(network, DEVICE)
fast_bot  = make_bot(game, evaluator, FAST_MCTS_SIMS)
full_bot  = make_bot(game, evaluator, FULL_MCTS_SIMS)

replay_buffer = []
elo           = 1000.0
snapshot      = copy.deepcopy(network).to(DEVICE)
hist = {'ep': [], 'p_loss': [], 'v_loss': [],
        'vs_random': [], 'vs_prev': [], 'elo': []}

n_params = sum(p.numel() for p in network.parameters())
print(f'Device: {DEVICE}  |  BoopNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | 8× augmentation')
print('-' * 72)

for ep in range(1, NUM_EPISODES + 1):

    # 1. Self-play with playout cap randomization
    network.eval()
    bot      = fast_bot if random.random() < FAST_GAME_PROB else full_bot
    raw_data = self_play_game(game, bot, temp_threshold=TEMP_THRESHOLD)

    # 2. Augment each move with all 8 board symmetries (8× free data)
    for obs, pol, val in raw_data:
        replay_buffer.extend(augment_sample(obs, pol, val))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train
    network.train()
    p_losses, v_losses = [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch  = random.sample(replay_buffer, BATCH_SIZE)
            x_b    = batch_to_tensor([s[0] for s in batch], DEVICE)
            pol_b  = torch.from_numpy(
                         np.stack([s[1] for s in batch]).astype(np.float32)
                     ).to(DEVICE)
            val_b  = torch.from_numpy(
                         np.array([[s[2]] for s in batch], dtype=np.float32)
                     ).to(DEVICE)

            logits, value = network(x_b)
            log_p  = F.log_softmax(logits, dim=1)

            # Cross-entropy vs soft MCTS policy targets
            p_loss = -(pol_b * log_p).sum(dim=1).mean()
            # Entropy bonus: keeps policy spread, prevents mode collapse
            entropy = -(torch.exp(log_p) * log_p).sum(dim=1).mean()
            # Value MSE; both tanh output and game returns are in [-1, 1]
            v_loss = F.mse_loss(value, val_b)

            loss = p_loss - ENTROPY_BONUS * entropy + v_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            p_losses.append(p_loss.item())
            v_losses.append(v_loss.item())
        scheduler.step()

    # 4. Periodic evaluation (greedy, no MCTS)
    if ep % EVAL_EVERY == 0:
        network.eval()
        wr_rand = eval_vs_random(game, network, EVAL_GAMES, DEVICE)
        wr_prev = eval_vs_snapshot(game, network, snapshot, EVAL_GAMES, DEVICE)
        elo     = update_elo(elo, RANDOM_ELO, wr_rand)
        snapshot = copy.deepcopy(network).to(DEVICE)

        mp = float(np.mean(p_losses)) if p_losses else float('nan')
        mv = float(np.mean(v_losses)) if v_losses else float('nan')
        hist['ep'].append(ep)
        hist['p_loss'].append(mp)
        hist['v_loss'].append(mv)
        hist['vs_random'].append(wr_rand)
        hist['vs_prev'].append(wr_prev)
        hist['elo'].append(elo)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {ep:5d} | p={mp:.3f} v={mv:.3f} | '
              f'vs_rand={wr_rand:.2f} vs_prev={wr_prev:.2f} | '
              f'ELO={elo:.0f} | lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Boop KataGo-Style Training', fontsize=14)

axes[0, 0].plot(hist['ep'], hist['p_loss'])
axes[0, 0].set_title('Policy Loss (cross-entropy)')
axes[0, 0].set_xlabel('Episode')

axes[0, 1].plot(hist['ep'], hist['v_loss'], color='tab:orange')
axes[0, 1].set_title('Value Loss (MSE)')
axes[0, 1].set_xlabel('Episode')

axes[0, 2].plot(hist['ep'], hist['vs_random'], color='tab:green')
axes[0, 2].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Win Rate vs Random (greedy)')
axes[0, 2].set_xlabel('Episode')
axes[0, 2].set_ylim(0, 1)

axes[1, 0].plot(hist['ep'], hist['vs_prev'], color='tab:purple')
axes[1, 0].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1, 0].set_title('Win Rate vs Previous Checkpoint')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylim(0, 1)

axes[1, 1].plot(hist['ep'], hist['elo'], color='tab:red')
axes[1, 1].axhline(RANDOM_ELO, color='gray', linestyle='--', linewidth=0.8,
                    label=f'Random baseline ({RANDOM_ELO:.0f})')
axes[1, 1].set_title('ELO Rating')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].legend()

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()